```
urbansound8k/
├── features.pkl   -- {sample_id: {"audio_path": <path_relative_to_dataset_root>}}
├── targets.pkl    -- {sample_id: {"class_id": <int>}}
├── train_keys.pkl -- {idx: {"key": sample_id}, ...}
├── val_keys.pkl   -- {idx: {"key": sample_id}, ...}
└── test_keys.pkl  -- {idx: {"key": sample_id}, ...}
```

In [1]:
import pickle
from pathlib import Path

import pandas as pd
import soundfile as sf
from datasets import load_dataset
from tqdm.auto import tqdm

<project_root>/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup

In [2]:
data_dir = Path(".").resolve().parent / "data"
raw_dir = data_dir / "raw" / "UrbanSound8K"
preprocessed_dir = data_dir / "preprocessed" / "urbansound8k"

val_folds = [9]
test_folds = [10]

print(f"Raw audio dir: {raw_dir}")
print(f"Preprocessed dir: {preprocessed_dir}")
print(f"Val folds: {val_folds}, Test folds: {test_folds}")

Raw audio dir: <project_root>/data/raw/UrbanSound8K
Preprocessed dir: <project_root>/data/preprocessed/urbansound8k
Val folds: [9], Test folds: [10]


## Download

In [3]:
ds = load_dataset("danavery/urbansound8K", split="train")
print(f"Loaded {len(ds)} samples")
print(f"Features: {ds.features}")

Loaded 8732 samples
Features: {'audio': Audio(sampling_rate=None, decode=True, num_channels=None, stream_index=None), 'slice_file_name': Value('string'), 'fsID': Value('int64'), 'start': Value('float64'), 'end': Value('float64'), 'salience': Value('int64'), 'fold': Value('int64'), 'classID': Value('int64'), 'class': Value('string')}


In [4]:
raw_dir.mkdir(parents=True, exist_ok=True)

for sample in tqdm(ds, desc="Saving audio files"):
    fold = sample["fold"]
    filename = sample["slice_file_name"]
    audio = sample["audio"]

    fold_dir = raw_dir / "audio" / f"fold{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    audio_path = fold_dir / filename
    if not audio_path.exists():
        sf.write(str(audio_path), audio["array"], audio["sampling_rate"])

Saving audio files: 100%|██████████| 8732/8732 [01:24<00:00, 103.33it/s]


## Build

In [ ]:
metadata = ds.to_pandas()[
    [
        "slice_file_name", 
        "fsID", 
        "start", 
        "end", 
        "salience", 
        "fold", 
        "classID", 
        "class"
    ]
]

print(f"Total samples: {len(metadata)}")
print(f"Classes:")
metadata["class"].value_counts().sort_index()

Total samples: 8732
Classes:


class
air_conditioner     1000
car_horn             429
children_playing    1000
dog_bark            1000
drilling            1000
engine_idling       1000
gun_shot             374
jackhammer          1000
siren                929
street_music        1000
Name: count, dtype: int64

In [7]:
features = {}
targets = {}

for _, row in metadata.iterrows():
    sample_id = row["slice_file_name"].replace(".wav", "")
    fold = row["fold"]
    audio_path = str(Path("audio") / f"fold{fold}" / row["slice_file_name"])

    features[sample_id] = {"audio_path": audio_path}
    targets[sample_id] = {"class_id": int(row["classID"])}

print(f"Features: {len(features)} samples")
print(f"Targets:  {len(targets)} samples")

Features: 8732 samples
Targets:  8732 samples


### train / val / test split

In [10]:
train_keys = {}
val_keys = {}
test_keys = {}

train_idx = val_idx = test_idx = 0

for _, row in metadata.iterrows():
    sample_id = row["slice_file_name"].replace(".wav", "")
    fold = row["fold"]

    if fold in test_folds:
        test_keys[test_idx] = {"key": sample_id}
        test_idx += 1
    elif fold in val_folds:
        val_keys[val_idx] = {"key": sample_id}
        val_idx += 1
    else:
        train_keys[train_idx] = {"key": sample_id}
        train_idx += 1

all_folds = set(metadata["fold"].unique().tolist())
train_folds = all_folds - set(val_folds) - set(test_folds)

print(f"Train folds: {sorted(train_folds)} -> {len(train_keys)} samples")
print(f"Val folds:   {val_folds} -> {len(val_keys)} samples")
print(f"Test folds:  {test_folds} -> {len(test_keys)} samples")

Train folds: [1, 2, 3, 4, 5, 6, 7, 8] -> 7079 samples
Val folds:   [9] -> 816 samples
Test folds:  [10] -> 837 samples


## Save preprocessed

In [11]:
preprocessed_dir.mkdir(parents=True, exist_ok=True)

for name, data in [
    ("features", features),
    ("targets", targets),
    ("train_keys", train_keys),
    ("val_keys", val_keys),
    ("test_keys", test_keys),
]:
    path = preprocessed_dir / f"{name}.pkl"
    with open(path, "wb") as f:
        pickle.dump(data, f)
    print(f"Saved {name}: {len(data)} entries -> {path}")

Saved features: 8732 entries -> <project_root>/data/preprocessed/urbansound8k/features.pkl
Saved targets: 8732 entries -> <project_root>/data/preprocessed/urbansound8k/targets.pkl
Saved train_keys: 7079 entries -> <project_root>/data/preprocessed/urbansound8k/train_keys.pkl
Saved val_keys: 816 entries -> <project_root>/data/preprocessed/urbansound8k/val_keys.pkl
Saved test_keys: 837 entries -> <project_root>/data/preprocessed/urbansound8k/test_keys.pkl
